# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark RDD - SOLUTION
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful

In [1]:
from pyspark import SparkContext, SparkConf
import numpy as np
import operator

In [2]:
conf=SparkConf().setAppName("Lab4-rdd").setMaster("local[*]")
sc = SparkContext(conf=conf)

Using PySpark and RDD's on the https://coding.csel.io machines is slow -- most of the code is executed in Python and this is much less efficient than the java-based code using the PySpark dataframes. Be patient and trying using `.cache()` to cache the output of joins. You may want to start with a reduced set of data before running the full task. You can use the `sample()` method to extract just a sample of the data or use 

These two RDD's are called "rawCitations" and "rawPatents" because you probably want to process them futher (e.g. convert them to integer types, etc). 

The `textFile` function returns data in strings. This should work fine for this lab.

Other methods you use might return data in type `Byte`. If you haven't used Python `Byte` types before, google it. You can convert a value of `x` type byte into e.g. a UTF8 string using `x.decode('uft-8')`. Alternatively, you can use the `open` method of the gzip library to read in all the lines as UTF-8 strings like this:
```
import gzip
with gzip.open('cite75_99.txt.gz', 'rt',encoding='utf-8') as f:
    rddCitations = sc.parallelize( f.readlines() )
```
This is less efficient than using `textFile` because `textFile` would use the underlying HDFS or other file system to read the file across all the worker nodes while the using `gzip.open()...readlines()` will read all the data in the frontend and then distribute it to all the worker nodes.

In [3]:
rddCitations = sc.textFile("cite75_99.txt.gz")
rddPatents = sc.textFile("apat63_99.txt.gz")

The data looks like the following.

In [4]:
rddCitations.take(5)

['"CITING","CITED"',
 '3858241,956203',
 '3858241,1324234',
 '3858241,3398406',
 '3858241,3557384']

In [5]:
rddPatents.take(5)

['"PATENT","GYEAR","GDATE","APPYEAR","COUNTRY","POSTATE","ASSIGNEE","ASSCODE","CLAIMS","NCLASS","CAT","SUBCAT","CMADE","CRECEIVE","RATIOCIT","GENERAL","ORIGINAL","FWDAPLAG","BCKGTLAG","SELFCTUB","SELFCTLB","SECDUPBD","SECDLWBD"',
 '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,',
 '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,',
 '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,',
 '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,']

In other words, they are a single string with multiple CSV's. You will need to convert these to (K,V) pairs, probably convert the keys to `int` and so on. You'll need to `filter` out the header string as well since there's no easy way to extract all the lines except the first.

# My Solution

## Step 1:
- Collect Patent Number and Postate. Patent Number to be used as Key and Postate used as value \
      * Patent Number is the first entry of each line \
      * Postate is the 6th entry of each line \
  **Skip the first line**


In [4]:
# Produces PATENT | POSTATE
patentRDD = rddPatents.zipWithIndex().filter(lambda row: row[1] > 0).keys()
# For each row, the first column is the patent number
# Postate is the 6th column
patentRDD = patentRDD.map(lambda value: (int(value.split(",")[0]), value.split(",")[5].strip('"')))

In [18]:
patentRDD.take(10)

[(3070801, ''),
 (3070802, 'TX'),
 (3070803, 'IL'),
 (3070804, 'OH'),
 (3070805, 'CA'),
 (3070806, 'PA'),
 (3070807, 'OH'),
 (3070808, 'IA'),
 (3070809, 'AZ'),
 (3070810, 'IL')]

## Step 2:
- Collect and split the citation txt data into a table like structure with the cited and citing patent numbers as integers \
  **SKIP FIRST LINE HEADERS**

In [5]:
# Produces: CITING | CITED
citationRDD = rddCitations.zipWithIndex().filter(lambda row: row[1] > 0) \
    .keys().map(lambda row: (int(row.split(",")[0]),int(row.split(",")[1])))

In [20]:
citationRDD.take(10)

[(3858241, 956203),
 (3858241, 1324234),
 (3858241, 3398406),
 (3858241, 3557384),
 (3858241, 3634889),
 (3858242, 1515701),
 (3858242, 3319261),
 (3858242, 3668705),
 (3858242, 3707004),
 (3858243, 2949611)]

In [6]:
# Create the CITED | CITING key,value pair so we can join on CITED
citedKeyRDD = citationRDD.map(lambda x: (x[1], x[0]))

In [7]:
# Join the key,value cited rdd with patent to produce CITED | (CITING | CITED_STATE)
citedJoin = citedKeyRDD.join(patentRDD)

In [8]:
citedJoin.cache()

PythonRDD[11] at RDD at PythonRDD.scala:53

In [34]:
citedJoin.take(10)

[(3167980, (3858423, 'CT')),
 (3454412, (3859234, 'MN')),
 (3454412, (3888821, 'MN')),
 (3454412, (4036811, 'MN')),
 (3454412, (4182382, 'MN')),
 (3454412, (4540727, 'MN')),
 (3454412, (4663371, 'MN')),
 (3613560, (3859908, 'MI')),
 (3613560, (3868903, 'MI')),
 (3613560, (3874282, 'MI'))]

In [9]:
# Now flip the CITED number and CITING number to produce CITING | (CITED | CITED_STATE)
# that will then be joined again with patent table to get the CITING state
citingJoin = citedJoin.map(lambda x: (x[1][0], (x[0], x[1][1])))

In [10]:
# join the patent table again to get CITING | ((CITED | CITED_STATE) | CITING_STATE)
intermediateTable = citingJoin.join(patentRDD)

In [11]:
intermediateTable.cache()

PythonRDD[19] at RDD at PythonRDD.scala:53

In [12]:
intermediateTable.take(5)

[(4774449, ((3970912, 'MD'), 'FL')),
 (4774449, ((4389608, 'CA'), 'FL')),
 (4774449, ((3176212, 'NJ'), 'FL')),
 (4774449, ((4383212, 'MO'), 'FL')),
 (4774449, ((3936718, ''), 'FL'))]

In [16]:
# Now remove the blank and non-matching state rows
filteredIntermediate = intermediateTable.filter( lambda x: x[1][0][1] != "" and x[1][1] != "" and x[1][0][1] == x[1][1])

In [17]:
filteredIntermediate.take(5)

[(4529136, ((4372496, 'CT'), 'CT')),
 (4529136, ((4127237, 'CT'), 'CT')),
 (4980693, ((4843400, 'CA'), 'CA')),
 (4980693, ((4740793, 'CA'), 'CA')),
 (4980693, ((4326203, 'CA'), 'CA'))]

In [20]:
# Produce a 'mapping/reducer' map style of table where you take each CITING number and place a 1 next to it in a new mapping
# because its already filtered so the contents have matching states
mapFiltered = filteredIntermediate.map(lambda x: (x[0], 1))

In [21]:
reducerFiltered = mapFiltered.reduceByKey(lambda x,y: x + y)

In [22]:
reducerFiltered.take(10)

[(4529136, 2),
 (4980693, 3),
 (5913036, 1),
 (4169181, 3),
 (5543067, 20),
 (5561322, 1),
 (4947639, 5),
 (5665953, 2),
 (5527041, 2),
 (5389356, 1)]

In [23]:
# Create the final table. Need to first create the original table just without the headers
patentFullRDD = rddPatents.zipWithIndex() \
    .filter(lambda row: row[1] > 0) \
    .keys() \
    .map(lambda row: (int(row.split(",")[0]), row))

In [24]:
patentFullRDD.take(5)

[(3070801, '3070801,1963,1096,,"BE","",,1,,269,6,69,,1,,0,,,,,,,'),
 (3070802, '3070802,1963,1096,,"US","TX",,1,,2,6,63,,0,,,,,,,,,'),
 (3070803, '3070803,1963,1096,,"US","IL",,1,,2,6,63,,9,,0.3704,,,,,,,'),
 (3070804, '3070804,1963,1096,,"US","OH",,1,,2,6,63,,3,,0.6667,,,,,,,'),
 (3070805, '3070805,1963,1096,,"US","CA",,1,,2,6,63,,1,,0,,,,,,,')]

In [25]:
finalRDD = patentFullRDD.leftOuterJoin(reducerFiltered)

In [26]:
finalRDD.take(5)

[(3071228, ('3071228,1963,1096,,"US","MI",,2,,192,5,53,,2,,0.5,,,,,,,', None)),
 (3071396, ('3071396,1963,1096,,"US","TX",,2,,285,6,67,,2,,0.5,,,,,,,', None)),
 (3071768,
  ('3071768,1963,1096,,"US","CA",,6,,343,2,21,,3,,0.4444,,,,,,,', None)),
 (3071936, ('3071936,1963,1103,,"US","NY",,1,,62,6,69,,1,,0,,,,,,,', None)),
 (3072084, ('3072084,1963,1103,,"IT","",,1,,112,6,63,,2,,0,,,,,,,', None))]

In [27]:
# Organize data and remove 'None' with a 0 instead
finalRDD = finalRDD.map( lambda x: (x[0], x[1][0], x[1][1] if x[1][1] is not None else 0) )

In [28]:
finalRDD.take(5)

[(3071228, '3071228,1963,1096,,"US","MI",,2,,192,5,53,,2,,0.5,,,,,,,', 0),
 (3071396, '3071396,1963,1096,,"US","TX",,2,,285,6,67,,2,,0.5,,,,,,,', 0),
 (3071768, '3071768,1963,1096,,"US","CA",,6,,343,2,21,,3,,0.4444,,,,,,,', 0),
 (3071936, '3071936,1963,1103,,"US","NY",,1,,62,6,69,,1,,0,,,,,,,', 0),
 (3072084, '3072084,1963,1103,,"IT","",,1,,112,6,63,,2,,0,,,,,,,', 0)]